# PARC2026 — Stage Prefetched Dataset from Google Drive

CPU runtimeでDriveへprefetch済みの `lerobot/libero_plus` を、A100 runtime開始後に **Driveから `/content` へ短時間でlocal staging** します。

学習をDrive直読みにはせず、local diskへコピーしてvideo I/O bottleneckを避けます。


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, time
subprocess.run(['nvidia-smi'], check=True)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/parc2026-cache/datasets/lerobot_libero_plus_v3_train')
LOCAL_ROOT=Path('/content/parc2026/datasets/public_libero_plus_v3_train')
marker=DRIVE_ROOT/'.parc_prefetch_complete.json'
if not marker.exists():
    raise FileNotFoundError(f'prefetch marker not found: {marker}')
manifest=json.loads(marker.read_text())
print('dataset:', manifest['dataset_id'])
print('revision:', manifest['revision'])
print('files:', manifest['required_file_count'])
print('size GiB:', round(manifest['total_bytes']/2**30,2))


## Drive source verification
completion manifestとDrive上の実ファイルを照合します。


In [ ]:
bad=[]
for row in manifest['files']:
    p=DRIVE_ROOT/row['path']
    if not p.exists() or p.stat().st_size != int(row['size_bytes']):
        bad.append(row['path'])
if bad:
    raise RuntimeError(f'Drive prefetch validation failed: {bad[:10]} total={len(bad)}')
print('DRIVE SOURCE GATE: PASS')


## Local staging
既に同サイズのfileがlocalにあればskipします。`.cache/huggingface` もコピーするため、Notebook 50の `hf_hub_download` は大容量再downloadではなくlocal hitになります。


In [ ]:
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
start=time.time()
copied=0
skipped=0
all_sources=[p for p in DRIVE_ROOT.rglob('*') if p.is_file()]
for i,src in enumerate(all_sources,1):
    rel=src.relative_to(DRIVE_ROOT)
    dst=LOCAL_ROOT/rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        skipped += 1
    else:
        shutil.copy2(src,dst)
        copied += 1
    if i==1 or i%10==0 or i==len(all_sources):
        print(f'stage {i}/{len(all_sources)} copied={copied} skipped={skipped}: {rel}')
print('stage wall sec:', round(time.time()-start,1))


## Local Gate
required filesがすべてlocalに同サイズで存在することを確認します。


In [ ]:
bad=[]
for row in manifest['files']:
    p=LOCAL_ROOT/row['path']
    if not p.exists() or p.stat().st_size != int(row['size_bytes']):
        bad.append(row['path'])
if bad:
    raise RuntimeError(f'local staging validation failed: {bad[:10]} total={len(bad)}')
shutil.copy2(marker, LOCAL_ROOT/'.parc_prefetch_complete.json')
print('LOCAL STAGE GATE: PASS')
print('local root:', LOCAL_ROOT)
print('Next: run 50_pi05_dataset_ablation.ipynb in this same runtime.')
